# Train & export — IsolationForest tourist-safety model

Trains on the Phase 10 synthetic dataset (`tools/seed-data/trajectories.csv`) when present,
otherwise on `app.synthetic` (5,000 normal + 500 anomalous windows).

IsolationForest is **unsupervised** — injected labels are used only for the precision / recall / F1 table you put on a slide.

Run from `services/ai` with the project venv activated so `app` is importable.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

from app.config import ARTIFACTS_DIR, CONTAMINATION, MODEL_META_PATH, MODEL_VERSION, anomaly_threshold
from app.features import FEATURE_NAMES
from app.models.isolation_forest import (
    decision_to_unit,
    export_onnx,
    export_scaler_json,
    load_model,
    save_sklearn,
    train_iforest,
)
from app.synthetic import load_or_generate
from app.train import _calibrate_onnx, _split, extract_matrix, _metrics

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print("model_version", MODEL_VERSION)
print("contamination", CONTAMINATION)
print("n_features", len(FEATURE_NAMES))
print("FEATURE_NAMES")
for i, name in enumerate(FEATURE_NAMES):
    print(f"  {i:2d}  {name}")

## 1. Load labelled windows

In [ ]:
windows, source = load_or_generate()
print(f"source={source}  n_windows={len(windows)}")
print(pd.Series([w.scenario for w in windows], name="scenario").value_counts())
print("labels", pd.Series([w.label for w in windows]).value_counts().to_dict())

## 2. Extract the frozen 18-feature vector

In [ ]:
X, y, scenarios = extract_matrix(windows)
train_idx, test_idx = _split(y)
X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]
scen_test = [scenarios[i] for i in test_idx]
print(X_train.shape, X_test.shape, "anomalies in test", int(y_test.sum()))

## 3. Fit IsolationForest (labels unused)

In [ ]:
pipeline = train_iforest(X_train)
threshold = anomaly_threshold()
d_train = pipeline.decision_function(X_train)
score_scale = float(np.median(d_train[d_train > 0]))
test_scores = decision_to_unit(
    pipeline.decision_function(X_test), threshold=threshold, scale=score_scale
)
y_pred = (test_scores >= threshold).astype(int)
print("threshold", threshold, "score_scale", score_scale)

## 4. Precision / recall / F1 (slide table)

In [ ]:
rows = [("HOLD-OUT", _metrics(y_test, y_pred))]
for name in sorted(set(scen_test)):
    mask = np.array([s == name for s in scen_test])
    rows.append((name, _metrics(y_test[mask], y_pred[mask])))

table = pd.DataFrame(
    [
        {
            "scenario": name,
            "precision": m["precision"],
            "recall": m["recall"],
            "f1": m["f1"],
            "support": int(m["support"]),
        }
        for name, m in rows
    ]
).set_index("scenario")
display(table.style.format({"precision": "{:.3f}", "recall": "{:.3f}", "f1": "{:.3f}"}))
print(table.to_markdown())

## 5. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=["normal", "anomaly"]
)
disp.plot(cmap="Blues")
disp.ax_.set_title("IsolationForest hold-out")
png = ARTIFACTS_DIR / "confusion_matrix.png"
disp.figure_.savefig(png, dpi=140, bbox_inches="tight")
print("wrote", png)

## 6. Export `iforest.onnx` + `scaler.json`

In [ ]:
scaler_spec = export_scaler_json(pipeline)
pkl = save_sklearn(pipeline)
onnx_path = export_onnx(pipeline, X_train[:8])
calib = _calibrate_onnx(pipeline, scaler_spec, X_test[:256])
print("pkl", pkl)
print("onnx", onnx_path)
print("scaler keys", list(scaler_spec))
print("onnx calibration", calib)

meta = {
    "model_version": MODEL_VERSION,
    "holdout": _metrics(y_test, y_pred),
    "dataset_source": source,
    **calib,
}
MODEL_META_PATH.write_text(json.dumps(meta, indent=2))
print("meta", MODEL_META_PATH)

## 7. sklearn vs ONNX (must agree within 1e-5)

In [ ]:
model = load_model()
sk = model.predict_sklearn(X_test[:64])
onx = model.predict_onnx(X_test[:64])
err = float(np.max(np.abs(sk.decision - onx.decision)))
print(f"max |Δ decision| = {err:.3e}")
assert err <= 1e-5, err
print("ok")